In [1]:
import os
import sys
import logging
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score



In [2]:
# Global Cache
DF_FEATURES_CACHE = None
SPLITS_CACHE = None
FEATURE_COLS_CACHE = None
DMATRICES_CACHE = None
LOG_LEVEL = logging.INFO
DATABASE_URI = "postgresql+psycopg2://USER@localhost:5432/eicu"
engine = create_engine(DATABASE_URI, future=True)
optimized_model_path = 'YOUR-PATH/actionable-hypotension/models_given/optimized/xgb_mix.json'

In [3]:
def load_model(model_path: str) -> xgb.Booster:
    bst = xgb.Booster()
    bst.load_model(model_path)
    return bst

In [4]:

# ----------------------
# Set up logging
# ----------------------
logging.basicConfig(
    level=LOG_LEVEL,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [5]:


def load_data(table_name: str) -> pd.DataFrame:
    """
    Load data from the specified SQL table.
    """
    
    query = f"SELECT * FROM public.{table_name}"
    df = pd.read_sql(query, engine)
    logging.info(f"Loaded {len(df)} rows, {df.shape[1]} columns")
    return df

def preprocess_data(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Drop metadata/time columns and encode the label.
    
    """
    drop_cols = [
        "patienthealthsystemstayid", "patientunitstayid",
        "context_start_offset_min", "context_end_offset_min",
        "target_start_offset_min", "target_end_offset"
    ]
    logging.info("Dropping metadata/time/JSON columns")
    df_features = df.drop(columns=[c for c in drop_cols if c in df.columns])


    # Label as int
    df_features["label"] = df_features["positive_event"].astype(int)
    
    if "admissionweight_bin" in df_features.columns:
        print("Found admissionweight_bin")
        df_features = df_features.rename(columns={"admissionweight_bin": "weight_bin"})
    if "admissionheight_bin" in df_features.columns:
        print("FOund admissionheight_bin")
        df_features = df_features.rename(columns={"admissionheight_bin": "height_bin"})

    return df_features
def get_feature_cols(df_features: pd.DataFrame) -> list:
    """
    Identify feature columns (exclude label and split markers).
    """
    excluded = {"positive_event", "positive_sample", "split", "label"}
    feature_cols = [c for c in df_features.columns if c not in excluded]
    
    # Log the remaining feature columns
    logging.info(f"Using {len(feature_cols)} feature columns: {feature_cols}")
    
    return feature_cols


def create_dmatrices(df, feature_cols):
    """
    Create XGBoost DMatrix objects for train, val, and test.
    """
    
    
    X = df[feature_cols]
    y = df["label"]
    logging.info(f"Creating DMatrix for df ({len(df)} rows)")
    dmatrix = xgb.DMatrix(X, label=y, missing=np.nan)
    return dmatrix

In [6]:
def prepare_testing_data_cached(table_name: str):
    global DF_FEATURES_CACHE, FEATURE_COLS_CACHE, DMATRICES_CACHE

    if all(v is not None for v in [DF_FEATURES_CACHE, FEATURE_COLS_CACHE, DMATRICES_CACHE]):
        logging.info("Using cached training data.")
        return DF_FEATURES_CACHE, FEATURE_COLS_CACHE, DMATRICES_CACHE

    logging.info(f"Loading testing data from public.{table_name}")
    
    df = load_data(table_name)

    # Vorverarbeitung
    df_features = preprocess_data(
        df
    )

    feature_cols = get_feature_cols(df_features)

    logging.info("✅ Training data prepared.")
    return df_features, feature_cols

In [7]:
df_features, feature_cols = prepare_testing_data_cached(table_name="merged_mix_features")

2026-02-16 20:15:29 [INFO] Loading testing data from public.merged_mix_features
2026-02-16 20:20:26 [INFO] Loaded 28941203 rows, 45 columns
2026-02-16 20:20:26 [INFO] Dropping metadata/time/JSON columns


Found admissionweight_bin
FOund admissionheight_bin


2026-02-16 20:20:30 [INFO] Using 36 feature columns: ['mean', 'median', 'min', 'max', 'std', 'iqr', 'first', 'last', 'rate_change', 'slope', 'weighted_mean', 'gender_bin', 'ethnicity_bin', 'age_bin', 'height_bin', 'weight_bin', 'bmi_bin', 'obesity', 'hypertension', 'diabetes', 'kidney_disease', 'lung_disease', 'heart_disease', 'drug_abuse', 'depression', 'sedatives_given', 'blood_products_transfusions_given', 'antibiotics_given', 'anticoagulants_antiplatelets_given', 'neuromuscular_blockers_given', 'analgesics_given', 'crystalloids_given', 'electrolytes_given', 'gi_protection_given', 'parenteral_nutrition_given', 'antiarrhythmics_given']
2026-02-16 20:20:30 [INFO] ✅ Training data prepared.


In [8]:
bst = load_model(optimized_model_path)
model_features = bst.feature_names

# Prüfen ob alle Features vorhanden sind:
missing_features = set(model_features) - set(df_features.columns)
if missing_features:
    logging.error(f"Fehlende Features: {missing_features}")

# DMatrix mit korrekten Features erstellen:
X_ext = df_features[model_features]
y_ext = df_features["label"]
dmatrix_ext = xgb.DMatrix(X_ext, label=y_ext)

preds = bst.predict(dmatrix_ext)

print("External AUC:", roc_auc_score(y_ext, preds))
print("External AUPRC:", average_precision_score(y_ext, preds))


External AUC: 0.604797700984277
External AUPRC: 0.0014107550630318043
